In [8]:
import pandas as pd
import numpy as np
import os
import gc
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

BASE_DIR = ".."
DATA_PATH = os.path.join(BASE_DIR, "data")
SUB_PATH = os.path.join(BASE_DIR, "submissions")

def reduce_mem(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        if df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')
    return df

print("Cell 1: Setup & Tools Ready!")

Cell 1: Setup & Tools Ready!


In [9]:
df_train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
df_test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

# Fix Battery & Signal Shifting
def align_data(df):
    # Ekstrak angka dari battery
    df['Battery_Level'] = df['Battery_Level'].astype(str).str.extract('(\d+)').astype(float)
    # Standarisasi Signal
    df['Signal_Strength'] = df['Signal_Strength'].astype(str).apply(
        lambda x: 'Low' if x in ['Edge', '3G', 'None'] else 'High'
    )
    return reduce_mem(df)

df_train = align_data(df_train)
df_test = align_data(df_test)
print("Cell 2: Alignment Done!")

Cell 2: Alignment Done!


In [10]:
def build_safe_features(df):
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['hour'] = df['Timestamp'].dt.hour
    
    # 1. Koordinat Kasar (Binning) - Pengganti Zona agar RAM Aman
    # Membulatkan koordinat ke 2 desimal (~1.1km)
    df['lat_bin'] = df['Pickup_Lat'].round(2).astype(str)
    df['long_bin'] = df['Pickup_Long'].round(2).astype(str)
    
    # 2. Sensor Magnitude
    df['accel_mag'] = np.sqrt(df['Accel_X']**2 + df['Accel_Y']**2 + df['Accel_Z']**2)
    
    # 3. Nav Stress Signal (Sangat Ringan tapi Berguna)
    df['nav_stress'] = ((df['GPS_Accuracy_M'] > 18) & (df['Signal_Strength'] == 'Low')).astype(int)
    
    # 4. Route Deviation
    lat_diff = (df['Dropoff_Lat'] - df['Pickup_Lat']).abs()
    long_diff = (df['Dropoff_Long'] - df['Pickup_Long']).abs()
    euclidean = np.sqrt(lat_diff**2 + long_diff**2)
    df['route_dev'] = df['Distance_KM'] / (euclidean + 0.005)
    
    # 5. Device Brand
    df['Brand'] = df['Device_FP'].astype(str).apply(lambda x: x.split('-')[0])
    
    # DROP KOLOM BERAT: Zone, Device_FP, Timestamp, Trip_ID
    cols_to_drop = ['Timestamp', 'Pickup_Zone', 'Dropoff_Zone', 'Device_FP']
    if 'Trip_ID' in df.columns: cols_to_drop.append('Trip_ID')
    
    return df.drop(columns=cols_to_drop)

y = df_train['Trip_Label']
X = build_safe_features(df_train.drop(columns=['Trip_Label']))
X_test_final = build_safe_features(df_test)

# LANGSUNG HAPUS DF ASLI
del df_train
gc.collect()
print("Cell 3: Fitur Selesai & RAM Dibersihkan!")

Cell 3: Fitur Selesai & RAM Dibersihkan!


In [11]:
# 1. Ambil index secara acak (Strategi Hemat RAM)
total_len = len(X)
train_size = int(0.08 * total_len) 
val_size = int(0.02 * total_len)  

indices = np.random.permutation(total_len)
train_idx = indices[:train_size]
val_idx = indices[train_size:train_size + val_size]

# 2. Potong langsung
X_train = X.iloc[train_idx].copy()
y_train = y.iloc[train_idx].copy()
X_val = X.iloc[val_idx].copy()
y_val = y.iloc[val_idx].copy()

# SEGERA HAPUS X dan y asli agar RAM lega
del X, y
gc.collect()

# 3. Handling Kategorikal & DEFINISIKAN cat_features
cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

for col in cat_features:
    X_train[col] = X_train[col].fillna("UNK").astype(str)
    X_val[col] = X_val[col].fillna("UNK").astype(str)
    X_test_final[col] = X_test_final[col].fillna("UNK").astype(str)

gc.collect()
print(f"Cell 4 (Fix): cat_features terdeteksi: {cat_features}")
print(f"Ukuran: Train {len(X_train)}, Val {len(X_val)}")

Cell 4 (Fix): cat_features terdeteksi: ['Promo_Code', 'Car_Model', 'Payment_Method', 'Weather', 'Traffic', 'Signal_Strength', 'lat_bin', 'long_bin', 'Brand']
Ukuran: Train 640000, Val 160000


In [12]:
model = CatBoostClassifier(
    iterations=600,       # Persingkat iterasi
    learning_rate=0.1,    # Percepat belajar
    depth=5,              # Turunkan ke 5 (RAM laptop bakal sangat lega)
    task_type="CPU",
    thread_count=8,       # Pakai 8 thread saja agar sisa buat sistem
    loss_function='MultiClass',
    # Matikan auto_class_weights agar RAM tidak spike
    early_stopping_rounds=30,
    random_seed=42,
    verbose=50
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    use_best_model=True
)

0:	learn: 1.4119013	test: 1.4114551	best: 1.4114551 (0)	total: 1.59s	remaining: 15m 50s
50:	learn: 0.6360774	test: 0.6347816	best: 0.6347816 (50)	total: 1m 15s	remaining: 13m 27s
100:	learn: 0.6229090	test: 0.6217215	best: 0.6217215 (100)	total: 2m 29s	remaining: 12m 18s
150:	learn: 0.6202879	test: 0.6193092	best: 0.6193092 (150)	total: 3m 45s	remaining: 11m 11s
200:	learn: 0.6187278	test: 0.6181010	best: 0.6181010 (200)	total: 5m 5s	remaining: 10m 7s
250:	learn: 0.6176440	test: 0.6172809	best: 0.6172809 (250)	total: 6m 27s	remaining: 8m 58s
300:	learn: 0.6169470	test: 0.6168982	best: 0.6168946 (298)	total: 7m 47s	remaining: 7m 44s
350:	learn: 0.6163453	test: 0.6165958	best: 0.6165958 (350)	total: 9m 7s	remaining: 6m 28s
400:	learn: 0.6159513	test: 0.6165029	best: 0.6165029 (400)	total: 10m 27s	remaining: 5m 11s
450:	learn: 0.6154720	test: 0.6163802	best: 0.6163778 (447)	total: 11m 48s	remaining: 3m 54s
500:	learn: 0.6149775	test: 0.6162131	best: 0.6162110 (486)	total: 13m 9s	remaining

In [13]:
# Ambil Probabilitas, bukan Label langsung
probs = model.predict_proba(X_val)
class_names = model.classes_

# Beri "Bonus" pengali untuk kelas minoritas agar Recall naik
# Index bisa berbeda, sesuaikan dengan urutan class_names
# Navigation_Issue dan Service_Complaint biasanya ada di index tertentu
multipliers = {
    'Navigation_Issue': 1.4,
    'Service_Complaint': 1.4,
    'Fraud_Indication': 1.2,
    'Perfect_Trip': 0.95,
    'Safety_Violation': 1.0
}

# Terapkan pengali ke masing-masing kolom probabilitas
for i, label in enumerate(class_names):
    if label in multipliers:
        probs[:, i] *= multipliers[label]

# Ambil prediksi baru berdasarkan probabilitas yang sudah di-adjust
y_val_pred = [class_names[np.argmax(p)] for p in probs]

print("HASIL EVALUASI DENGAN PROBABILITY SHIFTING:")
print(f"Macro F1: {f1_score(y_val, y_val_pred, average='macro'):.4f}")
print("-" * 50)
print(classification_report(y_val, y_val_pred))

HASIL EVALUASI DENGAN PROBABILITY SHIFTING:
Macro F1: 0.5570
--------------------------------------------------


/home/mmarufqk/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mmarufqk/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                   precision    recall  f1-score   support

 Fraud_Indication       0.97      0.98      0.97      7947
 Navigation_Issue       0.00      0.00      0.00     16095
     Perfect_Trip       0.72      1.00      0.84     88027
 Safety_Violation       0.99      0.95      0.97     32180
Service_Complaint       0.00      0.00      0.00     15751

         accuracy                           0.79    160000
        macro avg       0.54      0.59      0.56    160000
     weighted avg       0.65      0.79      0.71    160000



/home/mmarufqk/.local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
test_probs = model.predict_proba(X_test_final)

for i, label in enumerate(class_names):
    if label in multipliers:
        test_probs[:, i] *= multipliers[label]

test_preds = [class_names[np.argmax(p)] for p in test_probs]

submission = pd.DataFrame({
    "Trip_ID": df_test["Trip_ID"],
    "Trip_Label": test_preds
})
submission.to_csv(os.path.join(SUB_PATH, "submission_final_06_v2.csv"), index=False)
print("Bismillah, Silakan Submit dan Cek Skor Kaggle Kamu!")

Bismillah, Silakan Submit dan Cek Skor Kaggle Kamu!
